In [1]:
#1 : imports + Spark
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

from mastodon_utils import get_spark, read_table, write_table, DATA_DIR

spark = get_spark("mastodon-sentiment")

Spark 3.5.3 prêt - Spark UI : http://localhost:4041


In [2]:
#2 : charger Sentiment140
s140_schema = StructType([
    StructField("target", IntegerType()),
    StructField("id", StringType()),
    StructField("date", StringType()),
    StructField("flag", StringType()),
    StructField("user", StringType()),
    StructField("text", StringType()),
])

tweets = (
    spark.read.csv(
        f"{DATA_DIR}/training.1600000.processed.noemoticon.csv",
        schema=s140_schema, encoding="ISO-8859-1",
    )
    .select(F.when(F.col("target") == 4, 1.0).otherwise(0.0).alias("label"), "text")
)

SAMPLE_FRACTION = 0.2
tweets = tweets.sample(fraction=SAMPLE_FRACTION, seed=42)

tweets.groupBy("label").count().show()

+-----+------+
|label| count|
+-----+------+
|  0.0|160327|
|  1.0|160432|
+-----+------+



In [3]:
#3 : nettoyer le texte
def clean_text(col):
    c = F.lower(col)
    c = F.regexp_replace(c, r"https?://\S+", " ")# liens
    c = F.regexp_replace(c, r"@\w+", " ") # mentions @user
    c = F.regexp_replace(c, r"#", " ")  # on garde le mot du hashtag, sans le #
    c = F.regexp_replace(c, r"[^a-z' ]", " ") # on ne garde que les lettres
    c = F.regexp_replace(c, r"\s+", " ")
    return F.trim(c)

tweets = tweets.withColumn("clean_text", clean_text(F.col("text"))).filter(F.length("clean_text") > 0)
tweets.show(5, truncate=80)

+-----+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|label|                                                                            text|                                                                      clean_text|
+-----+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|  0.0|@LOLTrish hey  long time no see! Yes.. Rains a bit ,only a bit  LOL , I'm fin...|   hey long time no see yes rains a bit only a bit lol i'm fine thanks how's you|
|  0.0|Hollis' death scene will hurt me severely to watch on film  wry is directors ...|hollis' death scene will hurt me severely to watch on film wry is directors c...|
|  0.0|                @LettyA ahh ive always wanted to see rent  love the soundtrack!!|                           ahh ive always wanted to see rent l

In [4]:
#4 : séparer entraînement et test
train, test = tweets.randomSplit([0.8, 0.2], seed=42)
train.cache()
print("train :", train.count(), " | test :", test.count())

train : 256002  | test : 64025


In [5]:
#5 : le pipeline et l’entraînement
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
#on supprm les mots vides (the, a, etc..) pr contre  on garde les negations(not,cant,etc..), importantes pour le sentiment
stopwords = [w for w in StopWordsRemover.loadDefaultStopWords("english")
             if w not in ("not", "no", "nor", "don't", "isn't", "wasn't", "won't", "can't", "couldn't", "didn't")]
remover = StopWordsRemover(inputCol="words", outputCol="filtered", stopWords=stopwords)
vectorizer = CountVectorizer(inputCol="filtered", outputCol="tf", vocabSize=20000, minDF=5)
idf = IDF(inputCol="tf", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20, regParam=0.1)

pipeline = Pipeline(stages=[tokenizer, remover, vectorizer, idf, lr])
model = pipeline.fit(train)   # entrainement 2/4 min

In [6]:
#6 : évaluation
predictions = model.transform(test)

accuracy = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(predictions)
auc = BinaryClassificationEvaluator(labelCol="label").evaluate(predictions)
print(f"Accuracy : {accuracy:.3f} | F1 : {f1:.3f} | AUC : {auc:.3f}")

#matrice de confusion
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

Accuracy : 0.776 | F1 : 0.776 | AUC : 0.848
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|24305|
|  0.0|       1.0| 7767|
|  1.0|       0.0| 6543|
|  1.0|       1.0|25410|
+-----+----------+-----+



In [10]:
#7 : sauvegarder le modèle
model.write().overwrite().save(f"{DATA_DIR}/models/sentiment_lr")

In [11]:
#8 : appliquer le modèle aux toots Mastodon
toots = (
    read_table(spark, "toots")
    .dropDuplicates(["toot_id"]) 
    .filter(F.col("language") == "en")  
    .withColumn("clean_text", clean_text(F.col("text")))
    .filter(F.length("clean_text") > 0)
)

scored = (
    model.transform(toots)
    .withColumn("prob_positive", vector_to_array("probability")[1])

    .withColumn(
        "sentiment",
        F.when(F.col("prob_positive") >= 0.65, "positive")
         .when(F.col("prob_positive") <= 0.35, "negative")
         .otherwise("neutral"),
    )
    .select("toot_id", "event_time", "username", "text", "hashtags",
            F.round("prob_positive", 3).alias("prob_positive"), "sentiment")
)

scored.groupBy("sentiment").count().orderBy("sentiment").show()
scored.select("sentiment", "prob_positive", "text").show(10, truncate=90)

+---------+-----+
|sentiment|count|
+---------+-----+
| negative|  236|
|  neutral|  290|
| positive|  504|
+---------+-----+

+---------+-------------+------------------------------------------------------------------------------------------+
|sentiment|prob_positive|                                                                                      text|
+---------+-------------+------------------------------------------------------------------------------------------+
| positive|        0.899|#ApacheSpark with #Scala Online #training is planned in a way such that you progress in...|
| positive|        0.762|🔥 Hot off the press: “Softinio's Notes on Software Engineering - Issue #5” http://news...|
|  neutral|         0.52|There is nothing public yet, but we are getting ready to hire in #India (Pune, I suspec...|
|  neutral|        0.602|Referenced link: https://hackernoon.com/scale-vision-transformers-vit-beyond-hugging-fa...|
|  neutral|        0.602|Referenced link: https://hacke

In [12]:
#9 : enregistrer dans PostgreSQL
write_table(scored, "toots_sentiment", mode="overwrite")
print("table toots_sentiment écrite :", scored.count(), "toots")

table toots_sentiment écrite : 1030 toots
